# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AIz-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"    
MODEL = 'gemini-2.5-flash'
openai = OpenAI(base_url=BASE_URL, api_key=api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [13]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    #return result
    links = json.loads(result)
    return links
    

In [14]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [15]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [16]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-2.5-flash
Found 6 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'project/product page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project/product page',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'related company/product',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'patent information',
   'url': 'https://patents.google.com/patent/US20210049536A1/'}]}

In [17]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 13 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'product page', 'url': 'https://huggingface.co/models'},
  {'type': 'product page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand information page', 'url': 'https://huggingface.co/brand'},
  {'type': 'learning resources page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'social media', 'url': 'https://github.com/huggingface'},
  {'type': 'social media', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'social media', 'url': 'https://www.zhihu.com/org/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [19]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 10 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ models
Trending on
this week
Models
Tongyi-MAI/Z-Image-Turbo
Updated
3 days ago
•
44.5k
•
1.5k
black-forest-labs/FLUX.2-dev
Updated
4 days ago
•
171k
•
764
tencent/HunyuanOCR
Updated
4 days ago
•
92.9k
•
547
deepseek-ai/DeepSeek-Math-V2
Updated
4 days ago
•
3.53k
•
513
microsoft/Fara-7B
Updated
2 days ago
•
10.7k
•
330
Browse 1M+ models
Spaces
Running
on
Zero
308
FLUX.2 [dev]
💻
308
Generate images from text prompts with optional image editing
Running
on
Zero
MCP
Featured
1.42k
Qwen Image Edit Camera Control
🎬
1.42k
Fast 4 step inference with Qwen Image Edit 2509


In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nTongyi-MAI/Z-Image-Turbo\nUpdated\n3 days ago\n•\n44.5k\n•\n1.5k\nblack-forest-labs/FLUX.2-dev\nUpdated\n4 days ago\n•\n171k\n•\n764\ntencent/HunyuanOCR\nUpdated\n4 days ago\n•\n92.9k\n•\n547\ndeepseek-ai/DeepSeek-Math-V2\nUpdated\n4 days ago\n•\n3.53k\n•\n513\nmicrosoft/Fara-7B\nUpdated\n2 days ago\n•\n10.7k\n•\n330\nBrowse 1M+ models\nSpaces\nRunning\non\nZero\n308\nFLUX.2 [dev]\n💻\n308\n

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 9 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


**Hugging Face: The AI Community Building the Future**

Welcome to Hugging Face, the leading platform where the global machine learning community comes together to collaborate, innovate, and build the future of artificial intelligence. We empower individuals, teams, and enterprises to create, discover, and accelerate their ML journey like never before.

**What We Offer:**

*   **Models:** Explore and utilize over 1 million pre-trained models across various AI tasks, from state-of-the-art language models to advanced image generation and beyond.
*   **Datasets:** Access a vast collection of over 250,000 datasets to train your models, ensuring quality and diversity for your projects.
*   **Spaces:** Deploy and share your AI applications with the world. Our platform hosts over 400,000 interactive applications, allowing you to showcase your work and collaborate effortlessly.
*   **Open Source Stack:** Leverage our powerful open-source tools to move faster and build cutting-edge ML solutions.
*   **Explore All Modalities:** Work seamlessly with text, image, video, audio, and even 3D data, pushing the boundaries of what's possible in AI.

**The Home of Machine Learning Collaboration:**

Hugging Face is designed for collaboration. Host and share unlimited public models, datasets, and applications, and build a robust ML portfolio that highlights your expertise. Whether you're a seasoned researcher or an aspiring ML engineer, our platform provides the tools and community to support your growth.

**Accelerate Your Enterprise ML:**

For organizations seeking to scale their AI initiatives, Hugging Face offers robust Team and Enterprise solutions:

*   **Team:** Starting at just $20/user/month, give your team access to an advanced platform to build AI collaboratively.
*   **Enterprise:** For larger organizations, contact our sales team to explore flexible contract options, offering enterprise-grade security, access controls, dedicated support, Single Sign-On (SSO), configurable data regions, and comprehensive audit logs to maintain full control.

Join the millions building the future of AI. Discover, create, and collaborate with Hugging Face today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 16 relevant links


**Hugging Face: The AI Community Building the Future**

Welcome to Hugging Face, the leading platform where the global machine learning community collaborates to build the future of AI. We are the home for creating, discovering, and sharing models, datasets, and applications, accelerating innovation across all modalities.

**What We Offer:**

*   **Models:** Explore and contribute to over 1 million diverse AI models, from cutting-edge research to practical applications.
*   **Datasets:** Access and utilize more than 250,000 datasets to train and evaluate your machine learning projects.
*   **Spaces:** Deploy and showcase your AI applications with ease. Discover over 400,000 applications, transforming ideas into interactive experiences.

Whether you work with text, image, video, audio, or 3D data, Hugging Face provides the tools and community to bring your projects to life. Our platform is built on an open-source stack, empowering you to move faster and explore the vast landscape of machine learning.

**For Individuals & Community:**
Join a vibrant community of ML enthusiasts and experts. Host unlimited public models, datasets, and applications for free, build your professional ML portfolio, and share your work with the world. It’s the perfect place to learn, contribute, and collaborate.

**For Teams & Enterprises:**
Accelerate your organization's AI development with our robust paid solutions designed for scalability, security, and dedicated support.

*   **Team:** Starting at $20/user/month, our Team subscription provides advanced collaboration features for smaller groups.
*   **Enterprise:** For larger organizations, contact our sales team to explore flexible contract options tailored to your specific needs.

Our enterprise solutions offer:
*   **Enterprise-grade Security & Access Controls:** Protect your intellectual property with enhanced security measures.
*   **Single Sign-On (SSO):** Seamlessly integrate with your existing identity providers.
*   **Regional Data Control:** Select, manage, and audit the location of your repository data for compliance and performance.
*   **Comprehensive Audit Logs:** Maintain full visibility and control over actions taken within your organization's account.
*   **Dedicated Support:** Receive expert assistance to ensure your team's success.

Hugging Face is more than just a platform; it's a collaborative ecosystem dedicated to making machine learning accessible and powerful for everyone. Join us and be part of the community building the future of AI.

In [29]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:
# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 6 relevant links


# Hugging Face: We're Giving AI a Hug (And a Rocket Boost!)

Ever thought AI felt a little... distant? Like a super-smart robot that forgot its manners? Not anymore! At Hugging Face, we're not just building the future of AI; we're giving it a big, warm, collaborative cuddle. We're the platform where machine learning gets friendly, fast, and fantastically powerful.

### What We Do: Your Brain, But For AI!

Imagine a digital playground overflowing with **over 1 Million Models**, **250,000 Datasets**, and a staggering **400,000 Applications**. Yeah, we know, that's a lot of brainpower! Whether you're teaching a computer to tell a cat from a particularly fluffy cloud, generating art that defies reality, or building the next big thing in video analysis, we've got the tools. We're like a buffet for machine learning, and everything is open-source, collaborative, and absolutely delicious for your projects across text, image, video, audio, and even 3D. Move faster than a caffeinated coder on a deadline with our open-source stack, and build a portfolio that'll make even the most stoic AI gasp!

### For the Grown-Ups: Your AI, But On Steroids (Safely)

Got a team? Got an entire enterprise to conquer? Our **Team & Enterprise Hub** is your secret weapon. Scale your organization's AI dreams from a whisper to a thunderous roar with enterprise-grade security, dedicated support, single sign-on (because remembering passwords is *so* last century), and audit logs. We keep your data tucked in safely in the region of your choice, ensuring no rogue algorithms run amok on your watch.

### Our Culture: Democratizing Genius, One Commit at a Time

Our mission? To **democratize good machine learning, one commit at a time**. We're a diverse, passionate crew of humans (and perhaps a few sentient algorithms, we don't ask) who believe everyone deserves a shot at building something amazing with AI. We foster a vibrant community where collaboration isn't just a buzzword, it's our superpower. We're serious about making ML accessible and powerful, but we're pretty sure a sense of humor helps us get there faster.

### Join the Hugging Crew: Your Future (and AI's) Awaits!

If the idea of giving AI a good cuddle (and occasionally a stern, data-driven talking-to) excites you, then why aren't you **joining us**? We're constantly pushing boundaries, publishing groundbreaking papers, and making history. If "democratizing good machine learning" sounds like your calling, well, our virtual door is always open. Come make your mark, one clever line of code, one brilliant idea, one perfectly organized dataset at a time.

**Ready to explore?** Sign up, browse models, or just say hello! Your journey to building the future (and getting a digital hug) starts at Hugging Face.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>